In [37]:
import dolfinx
print(dolfinx.__version__)

# resoure from https://comet-fenics.readthedocs.io/en/latest/demo/modal_analysis_dynamics/cantilever_modal.html

0.11.0


In [38]:
import numpy as np
from dolfinx import mesh,fem
from mpi4py import MPI
import ufl

In [39]:
L,B,H = 20.0, 0.5, 1.0
nx = 200
ny,nz = int(B/L*nx)+1, int(H/L*nx) +1
print(nx,ny,nz)

200 6 11


In [40]:
domain = mesh.create_box(MPI.COMM_WORLD,[[0,0,0],[L,B,H]],[nx,ny,nz])


In [41]:
E,nu = fem.Constant(domain,1e5), fem.Constant(domain,0.0)
rho = fem.Constant(domain,1e-3)
mu = E / (2.0*(1+nu))
lmbda = E*nu/((1 + nu)* (1 - 2*nu))

def eps(u):
    return ufl.sym(ufl.grad(u))

def sigma(u):
    return  (
        lmbda * ufl.tr(eps(u)) * ufl.Identity(3)
        + 2 * mu * eps(u)
    )   

In [42]:
V = fem.functionspace(domain,("Lagrange",1,(domain.geometry.dim,)))
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

tdim = domain.topology.dim
boundary_facets = mesh.locate_entities_boundary(domain, tdim - 1, lambda x: np.isclose(x[0], 0.0))
boundary_dofs = dolfinx.fem.locate_dofs_topological(V, tdim - 1, boundary_facets)


bc = fem.dirichletbc(
    value=fem.Constant(domain,(0.0,0.0,0.0)),   
    dofs=boundary_dofs, 
    V=V
)

In [45]:
k_form = fem.form(ufl.inner(sigma(u), eps(v)) * ufl.dx)
m_form = fem.form(rho * ufl.inner(u, v) * ufl.dx)

from dolfinx.fem import petsc

K = petsc.assemble_matrix(k_form, bcs=[bc])
K.assemble()

M = petsc.assemble_matrix(m_form, bcs=[bc])
M.assemble()

In [46]:
from slepc4py import SLEPc
eigensolver = SLEPc.EPS().create(MPI.COMM_WORLD)
eigensolver.setOperators(K, M)
eigensolver.setProblemType(SLEPc.EPS.ProblemType.GHEP)

eigensolver.setWhichEigenpairs(SLEPc.EPS.Which.SMALLEST_MAGNITUDE)
eigensolver.setDimensions(nev=4) # Ask for the first 6 modes
eigensolver.solve()

# --- 7. Extract Results ---
nconv = eigensolver.getConverged()
print(f"Number of converged eigenvalues: {nconv}")

for i in range(nconv):
    kr, ki = eigensolver.getEigenvalue(i)
    if ki == 0.0: # Check if eigenvalue is purely real
        omega = np.sqrt(kr)          # Angular frequency (rad/s)
        freq = omega / (2.0 * np.pi) # Cyclic frequency (Hz)
        print(f"Mode {i+1}: {freq:.2f} Hz")

Number of converged eigenvalues: 0
